# 02 — Cohort Runs: R6–R7

Math×code anchor for SaaS-Builder Stage-2 — Phase 3 Task 3.3 of the
notebooks-trio plan. Loads committed C2 + C3 verdicts and verifies:

- **R6**: softplus L¹ tightness (M2 pin)
- **R7**: HALT-on-flip safeguard for the S_t pin (Υ_t form
  INDISTINGUISHABLE; spec-pinned det+churn used downstream)

Verify-only: no PyMC re-fit, <30s wall. Tamper anchor surfaced on
every cell via `loader.trio_audit_sha256`.

In [1]:
from simulations.notebooks.saas_builder_stage_2.env import (
    DATA_ROOT, reproducibility_seed,
)
from simulations.saas_builder.verify import (
    CommittedArtifactLoader,
    verify_r6_softplus_l1_tightness,
    verify_r7_s_t_pin,
)

reproducibility_seed()
loader = CommittedArtifactLoader(DATA_ROOT)
print(f"Trio audit anchor: {loader.trio_audit_sha256}")

Trio audit anchor: 147e01dfcbda6a1b776beed403891ce5a1a5fefdd186c6dcf60e95e4b480e0c2


## §3.2 — R6: Softplus L¹ tightness (M2 pin, sticky-cost CLOSED)

**Decision-citation block**

- **Reference**: spec v1.2.1 §6.1 (sticky-cost form pin); `notes/STAGE_2_RESULTS.md` §3.2 line 163; M2 pin in `simulations/types/distributions.py`
- **Why**: TODO-COHORT-2 (sticky-cost form) was CLOSED at Stage-2 via the softplus regularizer with β ≈ 50/κ; the L¹ deviation < 1e-3·κ on [0, 2κ] is the M2 tightness gate
- **Relevance**: R6 confirms the M2 pin holds at the spec-prescribed β baseline
- **Connection**: TODO-COHORT-2 closure cited from `notes/SaaS_Builders_AI_NativeBuilders.md`; gates the cohort hedge cost form

$$
\bigl\| \mathrm{softplus}_\beta(x) - x_+ \bigr\|_{L^1[0, 2\kappa]} \;<\; 10^{-3} \cdot \kappa
\tag{R6}
$$

with $\beta = \beta_{\text{factor}} / \kappa$ and the spec-prescribed
baseline $\beta_{\text{factor}} = 50$. R6's verifier composes
``simulations.types.distributions.tightness_l1_deviation`` to compute
the deviation and gates against the threshold.

In [2]:
v6 = verify_r6_softplus_l1_tightness(
    kappa=1.0, tol_factor=1e-3, audit_sha256=loader.trio_audit_sha256,
)
assert v6.passed, v6.message
print(f"{v6.r_tag}: softplus L¹ deviation = {v6.actual:.6e}  "
      f"(threshold {v6.expected:.6e}; β_factor=50)")

R6: softplus L¹ deviation = 6.579637e-04  (threshold 1.000000e-03; β_factor=50)


The softplus regularizer with β ≈ 50/κ satisfies the M2 tightness pin
at κ = 1.0. TODO-COHORT-2 (sticky-cost form selection) is CLOSED in
favor of the softplus regularizer per `notes/SaaS_Builders_AI_NativeBuilders.md`
TODO closure log. The L¹ deviation is well below the 1e-3·κ threshold —
the cohort hedge cost form is well-conditioned.

## §3.3 — R7: S_t pin (Υ_t form INDISTINGUISHABLE; HALT-on-flip safeguard armed)

**Decision-citation block**

- **Reference**: spec v1.2.1 §6.1 lines 405–442 (S_t pin); `notes/STAGE_2_RESULTS.md` §3.3 line 180; verdict memo §6 (C3 INDISTINGUISHABLE)
- **Why**: C3 PSIS-LOO-CV produced INDISTINGUISHABLE (Δelpd/SE = 1.67 < 2.0); spec-pinned det+churn form survives via HALT-on-flip safeguard
- **Relevance**: R7's runtime-observable property is `revenue_form_verdict._metadata.halt_on_flip_comparison == True`; the spec-text S_t pin lives at prereg-lock §6.1 and is preserved iff the safeguard fires no flip
- **Connection**: TODO-COHORT-3 (Υ_t form) closure log entry: INDISTINGUISHABLE; spec-pinned det+churn used downstream at C4

**Spec-level pin** (`docs/specs/2026-05-07-saas-builder-stage-2-prereg-lock.md` §6.1):

$$
S_t \;=\; (1 - \lambda)^t, \qquad \lambda \sim \mathrm{Beta}(\alpha_S = 4.5,\; \beta_S = 95.5)
\tag{R7}
$$

**Runtime-observable safeguard** (`revenue_form_verdict.json::_metadata.halt_on_flip_comparison`):
True iff the C3 PSIS-LOO-CV comparison produced no flip vs the
legacy prior. R7 verifies this safeguard is armed; the spec-text
S_t form itself is preserved as long as the comparison did not flip.

In [3]:
revenue_verdict = loader.load_revenue_form_verdict()
v7 = verify_r7_s_t_pin(
    revenue_form_verdict=revenue_verdict,
    audit_sha256=loader.trio_audit_sha256,
)
assert v7.passed, v7.message
print(f"{v7.r_tag}: HALT-on-flip safeguard armed; "
      f"spec-pinned S_t = (1-λ)^t with λ ~ Beta(4.5, 95.5) preserved")

R7: HALT-on-flip safeguard armed; spec-pinned S_t = (1-λ)^t with λ ~ Beta(4.5, 95.5) preserved


The C3 verdict is INDISTINGUISHABLE (Δelpd/SE = 1.67 < 2.0 threshold);
the three Υ_t candidate forms (det+churn, ar1_log, martingale) are
not statistically distinguishable on the synthetic data. The
spec-pinned det+churn form is used downstream by C4. The HALT-on-flip
safeguard is armed: if a future re-run flipped the comparison vs
the legacy prior, the safeguard would fire and the spec-pinned form
would be re-locked. R7's pytest tests
(`test_r7_negative_halt_on_flip_false`,
`test_r7_negative_metadata_missing`) exercise the negative-case
gate.

### Figure: S_t survival curve under prior-mean λ + Beta(4.5, 95.5) prior

Top panel: the Beta(4.5, 95.5) prior density on λ. Bottom panel:
S_t = (1-λ)^t over t ∈ [0, 36] months evaluated at the prior mean
λ̂ = 4.5/100 = 0.045.


In [4]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import beta
from simulations.notebooks.saas_builder_stage_2.env import FIGURES_ROOT

FIGURES_ROOT.mkdir(exist_ok=True)
alpha_S, beta_S = 4.5, 95.5
lambda_grid = np.linspace(0, 0.20, 200)
prior_density = beta.pdf(lambda_grid, alpha_S, beta_S)
lambda_mean = alpha_S / (alpha_S + beta_S)
t_grid = np.linspace(0, 36, 200)
S_t = (1 - lambda_mean) ** t_grid

fig, axes = plt.subplots(2, 1, figsize=(6, 5))
axes[0].plot(lambda_grid, prior_density, "C2-", lw=1.5)
axes[0].axvline(lambda_mean, color="k", ls="--", lw=0.8,
                label=fr"prior mean $\hat\lambda={lambda_mean:.3f}$")
axes[0].set_xlabel(r"$\lambda$")
axes[0].set_ylabel(r"prior density $\mathrm{Beta}(4.5, 95.5)$")
axes[0].legend()
axes[0].set_title(r"$\lambda$ prior")

axes[1].plot(t_grid, S_t, "C0-", lw=1.5)
axes[1].set_xlabel("t (months)")
axes[1].set_ylabel(r"$S_t = (1-\hat\lambda)^t$")
axes[1].set_title("Survival under prior-mean $\\lambda$")
axes[1].set_ylim(0, 1.02)
fig.tight_layout()
fig.savefig(FIGURES_ROOT / "s_t_survival.pdf")
plt.close(fig)
print(f"figure written: {FIGURES_ROOT / 's_t_survival.pdf'}")


figure written: /home/jmsbpp/apps/abrigo-analytics/simulations/notebooks/saas_builder_stage_2/figures/s_t_survival.pdf


In [5]:
from simulations.saas_builder.verify import TrioRollup

verdicts = (v6, v7)
rollup = TrioRollup(
    verdicts=verdicts,
    all_passed=all(v.passed for v in verdicts),
    audit_sha256=loader.trio_audit_sha256,
)
print(f"Notebook 02 trio rollup: all_passed={rollup.all_passed}  "
      f"(R6–R7; {len(rollup.verdicts)} verdicts; "
      f"audit {rollup.audit_sha256[:8]}…)")
for v in rollup.verdicts:
    print(f"  {v.r_tag}: passed={v.passed}")

Notebook 02 trio rollup: all_passed=True  (R6–R7; 2 verdicts; audit 147e01df…)
  R6: passed=True
  R7: passed=True
